# ONIA · 5.5. Stopwords, stemming și lematizare

Material practic cu exemple independente. Pentru spaCy și Stanza sunt necesare instalarea și descărcarea modelelor românești.

**Scop:** să comparăm transformările lexicale și să alegem experimental varianta potrivită.

## Exemplul 1: Exemplul 1 — Stopwords cu păstrarea negațiilor

Exemplu independent, cu date sintetice.

In [ ]:
texte = ["Nu este bun produsul", "Este bun produsul"]
stopwords = {"este", "produsul", "și", "la"}
negatii = {"nu", "nici", "fără"}

for text in texte:
    tokeni = text.casefold().split()
    filtrati = [t for t in tokeni if t not in stopwords or t in negatii]
    print(text, "->", filtrati)


## Exemplul 2: Exemplul 2 — Frecvența documentelor, calculată manual

Exemplu independent, cu date sintetice.

In [ ]:
from collections import Counter

documente = [
    "elevii rezolvă probleme și scriu cod",
    "profesorii explică probleme de cod",
    "elevii învață limbajul python"
]
df = Counter()
for document in documente:
    termeni_distincti = set(document.casefold().split())
    df.update(termeni_distincti)

for termen, numar in sorted(df.items()):
    print(f"{termen:15s} apare în {numar} document(e)")


## Exemplul 3: Exemplul 3 — Vectorizare cu și fără stopwords

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documente = [
    "nu este bun acest produs",
    "este bun acest produs",
    "nu este recomandat acest produs"
]

stop = ["este", "acest"]
fara_filtru = CountVectorizer()
cu_filtru = CountVectorizer(stop_words=stop)

X0 = fara_filtru.fit_transform(documente)
X1 = cu_filtru.fit_transform(documente)
print("Vocabular inițial:", fara_filtru.get_feature_names_out())
print("După filtrare:", cu_filtru.get_feature_names_out())
print("Dimensiuni:", X0.shape, X1.shape)


## Exemplul 4: Exemplul 4 — Filtrare prin document frequency

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "tenis sport meci", "sport fotbal stadion",
    "sport înot bazin", "sport tenis turneu"
]
v = TfidfVectorizer(max_df=0.75, min_df=1)
X = v.fit_transform(corpus)
print(v.get_feature_names_out())
print(X.shape)


## Exemplul 5: Exemplul 5 — Demonstrație cu matricea document–termen

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["nu îmi place", "îmi place"]

v1 = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
v2 = CountVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    stop_words=["nu", "îmi"]
)

print(v1.fit_transform(texte).toarray())
print(v1.get_feature_names_out())
print(v2.fit_transform(texte).toarray())


## Exemplul 6: Exemplul 6 — SnowballStemmer pentru română

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
# Instalare: pip install nltk
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("romanian")
cuvinte = ["copil", "copilului", "copiii", "cărțile", "cărților", "frumoase"]
for cuvant in cuvinte:
    print(f"{cuvant:14s} -> {stemmer.stem(cuvant)}")


## Exemplul 7: Exemplul 7 — Comparăm vocabularul înainte și după stemming

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

corpus = ["elevii citesc cărți", "elevului îi plac cărțile", "elevii învață"]
stemmer = SnowballStemmer("romanian")
tokens = [d.casefold().split() for d in corpus]
vocab_initial = set(t for document in tokens for t in document)
vocab_stem = set(stemmer.stem(t) for document in tokens for t in document)
print("Inițial:", sorted(vocab_initial))
print("Stemming:", sorted(vocab_stem))
print("Dimensiuni:", len(vocab_initial), len(vocab_stem))


## Exemplul 8: Exemplul 8 — Lematizare cu spaCy

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
# Instalare în terminal:
# pip install spacy
# python -m spacy download ro_core_news_sm
import spacy

nlp = spacy.load("ro_core_news_sm")
text = "Copiii citeau cărți interesante și mergeau spre bibliotecă."
doc = nlp(text)
for tok in doc:
    if not tok.is_space:
        print(f"{tok.text:16s} {tok.pos_:8s} {tok.lemma_}")


## Exemplul 9: Exemplul 9 — Selectăm doar cuvintele de conținut

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
import spacy
nlp = spacy.load("ro_core_news_sm")

text = "Elevii talentați citesc cărți interesante în bibliotecă."
doc = nlp(text)
rezultat = [
    tok.lemma_.casefold()
    for tok in doc
    if tok.is_alpha and tok.pos_ in {"NOUN", "VERB", "ADJ"}
]
print(rezultat)


## Exemplul 10: Exemplul 10 — Stanza, pe propoziții și cuvinte

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
# Instalare: pip install stanza
import stanza
stanza.download("ro")  # o singură dată, când există acces la internet

nlp = stanza.Pipeline("ro", processors="tokenize,pos,lemma", use_gpu=False)
doc = nlp("Elevii au rezolvat problemele și au discutat soluțiile.")
for prop in doc.sentences:
    for cuvant in prop.words:
        print(f"{cuvant.text:15s} {cuvant.upos:7s} {cuvant.lemma}")


## Exemplul 11: Exemplul 11 — TF-IDF pe leme cu analyzer personalizat

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("ro_core_news_sm")
def leme(text):
    doc = nlp(text)
    return [tok.lemma_.casefold() for tok in doc if tok.is_alpha]

train = ["Elevii citesc cărțile", "Elevul citește manualele", "Copiii au cărți"]
vec = TfidfVectorizer(analyzer=leme)
X_train = vec.fit_transform(train)
X_nou = vec.transform(["Elevul a citit cartea"])
print(vec.get_feature_names_out())
print(X_train.shape, X_nou.shape)


## Exemplul 12: Exemplul 12 — Tokenizer personalizat, păstrând n-gramele de cuvinte

**Dependențe suplimentare:** codul trebuie rulat numai după instalarea pachetelor și a modelelor necesare.

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("ro_core_news_sm")
def tokenizer_lematizat(text):
    return [tok.lemma_.casefold() for tok in nlp(text) if tok.is_alpha]

vec = TfidfVectorizer(
    tokenizer=tokenizer_lematizat,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=1
)
X = vec.fit_transform(["Nu este bun", "Este bun produsul"])
print(vec.get_feature_names_out())


## Exemplul 13: Exemplul 13 — Comparație cu / fără stopwords, pe folduri

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = [
    "tenis meci sport", "sport fotbal stadion", "victorie tenis sport",
    "cod python algoritm", "python date cod", "programare algoritm date",
    "meci de fotbal", "turneu de tenis", "rezultat sportiv",
    "bibliotecă python", "date și cod", "problemă de programare"
]
y = [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for stop in [None, ["de", "și"]]:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=stop, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    scoruri = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro")
    print("Stopwords:", stop, "macro-F1 mediu:", round(scoruri.mean(), 3))


## Exemplul 14: Exemplul 14 — Configurarea Pipeline și GridSearchCV

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

X = ["tenis sport", "fotbal meci", "turneu de tenis", "sport echipă",
     "python date", "cod algoritm", "date python", "probleme algoritm"]
y = ["sport"] * 4 + ["tech"] * 4

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LinearSVC())
])
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__stop_words": [None, ["de"]],
    "model__C": [0.1, 1.0]
}
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
search = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1_macro")
search.fit(X, y)
print(search.best_params_)


## Exemplul 15: Exemplul 15 — Pipeline de sentiment cu recenzii

Exemplu independent, cu date sintetice.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

recenzii = [
    "produs foarte bun", "excelent și util", "îmi place foarte mult",
    "recomand acest produs", "calitate bună", "merită cumpărat",
    "produs foarte slab", "nu recomand", "nu îmi place",
    "calitate proastă", "nu este bun", "experiență dezamăgitoare"
]
etichete = [1] * 6 + [0] * 6

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(recenzii, etichete)
probe = ["nu recomand deloc", "foarte bun și util"]
for text, pred in zip(probe, model.predict(probe)):
    print(text, "->", "pozitiv" if pred == 1 else "negativ")


## Provocări pentru elevi

1. Compară 3 liste de stopwords, inclusiv una care elimină greșit negațiile.
2. Măsoară schimbarea vocabularului după stemming și lematizare.
3. Compară macro-F1 pe aceleași folduri pentru toate variantele.
4. Explică o eroare produsă de o transformare prea agresivă.